In [1]:

import pandas as pd
import numpy as np
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.ensemble import GradientBoostingRegressor

from sklearn.metrics import (
    r2_score, mean_absolute_error, mean_squared_error,
    accuracy_score, classification_report
)

# 1. Load dataset
df = sns.load_dataset("diamonds")

# 2. Clean data
df = df.dropna()
df = df[(df["x"] > 0) & (df["y"] > 0) & (df["z"] > 0)]

# 3. Features and target
X = df.drop(columns=["price"])
y = df["price"]

numeric_features = ["carat", "depth", "table", "x", "y", "z"]
categorical_features = ["cut", "color", "clarity"]

# 4. Preprocessing
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"),
     categorical_features)
])

# 5. Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 6. Regression models
regression_models = {
    "Linear Regression": LinearRegression(),

    "Decision Tree": DecisionTreeRegressor(
        max_depth=18, random_state=42
    ),

    "Random Forest": RandomForestRegressor(
        n_estimators=100, max_depth=22,
        n_jobs=-1, random_state=42
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=100, learning_rate=0.1,
        max_depth=3, random_state=42
    )
}

regression_results = {}
trained_regression = {}

print("\nREGRESSION MODEL COMPARISON")

for name, algorithm in regression_models.items():

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", algorithm)
    ])

    pipeline.fit(X_train, y_train)

    predictions = pipeline.predict(X_test)

    r2 = r2_score(y_test, predictions)
    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))

    regression_results[name] = {
        "R2 Score": r2,
        "MAE": mae,
        "RMSE": rmse
    }

    trained_regression[name] = pipeline

    print(f"\n{name}")
    print("R2 Score:", round(r2, 4))
    print("MAE:", round(mae, 2))
    print("RMSE:", round(rmse, 2))


# 7. Classification
# Divide prices into three categories using training-set quantiles
price_limits = y_train.quantile([1/3, 2/3]).values

def price_category(prices):
    return np.where(
        prices <= price_limits[0], "Budget",
        np.where(prices <= price_limits[1],
                 "Mid-range", "Luxury")
    )

y_class = pd.Series(price_category(y), index=y.index)

X_train_c = X.loc[X_train.index]
X_test_c = X.loc[X_test.index]

y_train_c = y_class.loc[X_train.index]
y_test_c = y_class.loc[X_test.index]

classification_models = {
    "Decision Tree": DecisionTreeClassifier(
        max_depth=15, random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=100, max_depth=20,
        n_jobs=-1, random_state=42
    ),

    "Logistic Regression": LogisticRegression(
        max_iter=1000
    )
}

classification_results = {}
trained_classification = {}

print("\nCLASSIFICATION MODEL COMPARISON")

for name, algorithm in classification_models.items():

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", algorithm)
    ])

    pipeline.fit(X_train_c, y_train_c)

    predictions = pipeline.predict(X_test_c)

    accuracy = accuracy_score(y_test_c, predictions)

    classification_results[name] = accuracy
    trained_classification[name] = pipeline

    print(f"\n{name}")
    print("Accuracy:", round(accuracy, 4))


# 8. Find best regression model
best_regression = max(
    regression_results,
    key=lambda name: regression_results[name]["R2 Score"]
)

best_classification = max(
    classification_results,
    key=classification_results.get
)

print("\nBest Regression Model:", best_regression)
print("Best Classification Model:", best_classification)

# 9. Save models and results
joblib.dump({
    "regression_models": trained_regression,
    "classification_models": trained_classification,
    "regression_results": regression_results,
    "classification_results": classification_results,
    "best_regression": best_regression,
    "best_classification": best_classification,
    "price_limits": price_limits
}, "diamond_models.pkl")

print("\nAll models saved successfully!")



REGRESSION MODEL COMPARISON

Linear Regression
R2 Score: 0.9195
MAE: 726.3
RMSE: 1135.94

Decision Tree
R2 Score: 0.9688
MAE: 341.21
RMSE: 707.04

Random Forest
R2 Score: 0.9829
MAE: 264.74
RMSE: 523.63

Gradient Boosting
R2 Score: 0.9671
MAE: 401.38
RMSE: 726.24

CLASSIFICATION MODEL COMPARISON

Decision Tree
Accuracy: 0.9461

Random Forest
Accuracy: 0.9554

Logistic Regression
Accuracy: 0.9546

Best Regression Model: Random Forest
Best Classification Model: Random Forest

All models saved successfully!


In [2]:
%%writefile diamond.py

import streamlit as st
import pandas as pd
import joblib
import seaborn as sns
import matplotlib.pyplot as plt

# -------------------------------
# PAGE CONFIGURATION
# -------------------------------

st.set_page_config(
    page_title="Diamond Price Intelligence",
    page_icon="💎",
    layout="wide"
)

# -------------------------------
# LOAD MODELS
# -------------------------------

@st.cache_resource
def load_models():
    return joblib.load("diamond_models.pkl")

data = load_models()

reg_models = data["regression_models"]
class_models = data["classification_models"]

reg_results = data["regression_results"]
class_results = data["classification_results"]

best_reg = data["best_regression"]
best_class = data["best_classification"]

# -------------------------------
# CUSTOM DESIGN
# -------------------------------

st.markdown("""
<style>
.stApp {
    background-color: #f5f7fb;
}
.main-title {
    font-size: 36px;
    font-weight: 800;
    color: #172554;
}
.subtitle {
    color: #64748b;
    font-size: 16px;
}
div[data-testid="stMetric"] {
    background-color: white;
    padding: 18px;
    border-radius: 12px;
    border: 1px solid #e2e8f0;
}
</style>
""", unsafe_allow_html=True)

# -------------------------------
# HEADER
# -------------------------------

st.markdown(
    '<div class="main-title">💎 Diamond Price Intelligence</div>',
    unsafe_allow_html=True
)

st.markdown(
    '<div class="subtitle">Machine Learning Based Diamond Valuation & Analysis</div>',
    unsafe_allow_html=True
)

st.divider()

# -------------------------------
# SIDEBAR
# -------------------------------

st.sidebar.title("💎 Navigation")

page = st.sidebar.radio(
    "Choose a section",
    ["Price Prediction", "Model Comparison", "About Project"]
)

st.sidebar.divider()

st.sidebar.info(
    "This application uses machine learning "
    "to estimate diamond prices and classify "
    "diamonds into price categories."
)

# -------------------------------
# PRICE PREDICTION
# -------------------------------

if page == "Price Prediction":

    st.subheader("🔍 Enter Diamond Details")

    col1, col2 = st.columns(2)

    with col1:

        st.markdown("### Physical Measurements")

        carat = st.slider(
            "Carat Weight",
            0.2, 5.0, 0.7, 0.01
        )

        depth = st.slider(
            "Depth Percentage",
            43.0, 79.0, 61.7, 0.1
        )

        table = st.slider(
            "Table Percentage",
            43.0, 79.0, 57.4, 0.1
        )

        x = st.slider(
            "Length (mm)",
            3.0, 11.0, 5.5, 0.01
        )

        y = st.slider(
            "Width (mm)",
            3.0, 11.0, 5.5, 0.01
        )

        z = st.slider(
            "Depth (mm)",
            1.0, 7.0, 3.4, 0.01
        )

    with col2:

        st.markdown("### Diamond Quality")

        cut = st.selectbox(
            "Cut Quality",
            ["Fair", "Good", "Very Good",
             "Premium", "Ideal"],
            index=4
        )

        color = st.selectbox(
            "Diamond Color",
            ["D", "E", "F", "G", "H", "I", "J"],
            index=3
        )

        clarity = st.selectbox(
            "Diamond Clarity",
            ["I1", "SI2", "SI1", "VS2",
             "VS1", "VVS2", "VVS1", "IF"],
            index=3
        )

        st.markdown("### Select Algorithm")

        regression_choice = st.selectbox(
            "Price Prediction Model",
            list(reg_models.keys()),
            index=list(reg_models.keys()).index(best_reg)
        )

        classification_choice = st.selectbox(
            "Price Category Model",
            list(class_models.keys()),
            index=list(class_models.keys()).index(best_class)
        )

    st.divider()

    # Input dataframe
    input_data = pd.DataFrame([{
        "carat": carat,
        "cut": cut,
        "color": color,
        "clarity": clarity,
        "depth": depth,
        "table": table,
        "x": x,
        "y": y,
        "z": z
    }])

    if st.button("💎 Predict Diamond Value", use_container_width=True):

        # Regression prediction
        predicted_price = reg_models[
            regression_choice
        ].predict(input_data)[0]

        # Classification prediction
        predicted_category = class_models[
            classification_choice
        ].predict(input_data)[0]

        st.success("Prediction completed successfully!")

        st.markdown("### 💰 Estimated Diamond Value")

        st.metric(
            "Predicted Price",
            f"${predicted_price:,.2f}"
        )

        col1, col2 = st.columns(2)

        with col1:
            st.info(
                f"**Price Category:** {predicted_category}"
            )

        with col2:
            st.info(
                f"**Regression Model:** {regression_choice}"
            )

        st.caption(
            "This is a machine learning estimate based on "
            "the training dataset, not a professional appraisal."
        )

# -------------------------------
# MODEL COMPARISON
# -------------------------------

elif page == "Model Comparison":

    st.subheader("📊 Machine Learning Model Comparison")

    st.markdown("### Regression Performance")

    regression_df = pd.DataFrame(
        reg_results
    ).T.reset_index()

    regression_df.columns = [
        "Model", "R2 Score", "MAE", "RMSE"
    ]

    st.dataframe(
        regression_df.style.format({
            "R2 Score": "{:.4f}",
            "MAE": "${:,.2f}",
            "RMSE": "${:,.2f}"
        }),
        use_container_width=True,
        hide_index=True
    )

    st.markdown("#### R² Score Comparison")

    fig, ax = plt.subplots(figsize=(9, 4))

    ax.bar(
        regression_df["Model"],
        regression_df["R2 Score"]
    )

    ax.set_ylabel("R² Score")
    ax.set_xlabel("Regression Models")
    ax.set_ylim(0, 1)
    ax.tick_params(axis="x", rotation=15)

    st.pyplot(fig)
    plt.close(fig)

    st.success(
        f"Best Regression Model: {best_reg}"
    )

    st.divider()

    st.markdown("### Classification Performance")

    classification_df = pd.DataFrame(
        list(class_results.items()),
        columns=["Model", "Accuracy"]
    )

    st.dataframe(
        classification_df.style.format({
            "Accuracy": "{:.4f}"
        }),
        use_container_width=True,
        hide_index=True
    )

    st.markdown("#### Accuracy Comparison")

    fig, ax = plt.subplots(figsize=(9, 4))

    ax.bar(
        classification_df["Model"],
        classification_df["Accuracy"]
    )

    ax.set_ylabel("Accuracy")
    ax.set_xlabel("Classification Models")
    ax.set_ylim(0, 1)
    ax.tick_params(axis="x", rotation=15)

    st.pyplot(fig)
    plt.close(fig)

    st.success(
        f"Best Classification Model: {best_class}"
    )

# -------------------------------
# ABOUT PROJECT
# -------------------------------

elif page == "About Project":

    st.subheader("📘 About This Project")

    st.write("""
    This project uses the Seaborn Diamonds dataset
    to build machine learning models.

    **Regression:**
    Predicts the estimated price of a diamond.

    **Classification:**
    Classifies diamonds into Budget, Mid-range,
    and Luxury price categories.

    **Algorithms used:**

    - Linear Regression
    - Decision Tree Regressor
    - Random Forest Regressor
    - Gradient Boosting Regressor
    - Decision Tree Classifier
    - Random Forest Classifier
    - Logistic Regression

    **Evaluation metrics:**

    - R² Score
    - Mean Absolute Error
    - Root Mean Squared Error
    - Classification Accuracy
    """)

    st.markdown("### Dataset Information")

    df = sns.load_dataset("diamonds")

    col1, col2, col3 = st.columns(3)

    col1.metric("Total Records", f"{len(df):,}")
    col2.metric("Total Features", len(df.columns) - 1)
    col3.metric("Target Variable", "Price")

    st.markdown("### Dataset Preview")

    st.dataframe(
        df.head(10),
        use_container_width=True
    )

st.divider()

st.caption(
    "Diamond Price Intelligence | Machine Learning Project"
)


Writing diamond.py
